In [ ]:
# ============================================================
# DRWEIBO SSEE ABLATION MULTI-SEED EXPERIMENT
#
# Sparse Conversation Structure Modeling
# for Early Rumor Verification
#
# Controlled ablation study:
#   GAT / w/o SEM / w/o AGCC / w/o ASE / Full SSEE
#
# V2 change:
#   Add feature-space alignment before adaptive fusion:
#
#   semantic_rep  -> Linear + LayerNorm -> h_x_aligned
#   structural_rep-> Linear + LayerNorm -> h_e_aligned
#
#   h_f = lambda * h_e_aligned + (1-lambda) * h_x_aligned
#
# IMPORTANT:
#   GAT / SEM / AGCC / ASE / data split / seed / optimizer
#   protocol are unchanged from the previous full-model run.
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import random
import time
from pathlib import Path
from collections import Counter, deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertModel,
    RobertaTokenizer,
    RobertaModel,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm import tqdm


# ============================================================
# 1. PATH CONFIG
# ============================================================

BASE_DIR = Path("/root/autodl-fs/processed_scsr")

SPLIT_DIR = BASE_DIR / "splits"

OUTPUT_DIR = (
    BASE_DIR /
    "experiment_results" /
    "full_model_v3"
)


# ============================================================
# 2. DATASET
# ============================================================

DATASET = "DRWeibo"

# Later:
# DATASET = "DRWeibo"

PHEME_FOLD = 1


# ============================================================
# 3. LOCAL PRETRAINED MODELS
# ============================================================

DRWEIBO_MODEL_DIR = (
    BASE_DIR /
    "chinese_roberta_wwm_ext"
)

PHEME_MODEL_DIR = (
    BASE_DIR /
    "roberta_base"
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

SEED = 42


# ============================================================
# 5. TEXT ENCODER
# ============================================================

MAX_LENGTH = 128
NODE_CHUNK_SIZE = 192


# ============================================================
# 6. BATCH
# ============================================================

BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 1


# ============================================================
# 7. GAT
# ============================================================

GAT_HIDDEN_DIM = 256
GAT_HEADS = 4

assert GAT_HIDDEN_DIM % GAT_HEADS == 0

GAT_DROPOUT = 0.3
GAT_ATTENTION_DROPOUT = 0.2
LEAKY_RELU_SLOPE = 0.2

MAKE_BIDIRECTIONAL = True


# ============================================================
# 8. SSEE
# ============================================================

SEM_EVIDENCE_DIM = 128
GRAPH_STAT_DIM = 6
SSEE_DROPOUT = 0.3


# ============================================================
# 9. SAF
# ============================================================

SPARSITY_HIDDEN_DIM = 64
SAF_DROPOUT = 0.2

# Hidden dimension of the representation-aware fusion gate
FUSION_GATE_HIDDEN_DIM = 128

# Common fusion-space dimensionality.
FUSION_DIM = GAT_HIDDEN_DIM


# ============================================================
# 10. OPTIMIZATION
# ============================================================

ENCODER_LR = 2e-5
NEW_MODULE_LR = 1e-3

WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
GRADIENT_CLIP = 1.0


# ============================================================
# 11. TRAINING
# ============================================================

MAX_EPOCHS = 10
PATIENCE = 3
NUM_WORKERS = 8


# ============================================================
# 12. RANDOM SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # RTX 4090 throughput settings.
    # Seeds remain fixed. TF32 is used consistently for all PHEME variants.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass


# ============================================================
# 13. DEVICE
# ============================================================

def get_device():

    if torch.cuda.is_available():

        device = torch.device("cuda")

        prop = torch.cuda.get_device_properties(0)

        total_memory = (
            prop.total_memory /
            1024 ** 3
        )

        print("\nCUDA available")
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Total VRAM: {total_memory:.2f} GB")

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

        return device

    print("\nCUDA unavailable. Using CPU.")

    return torch.device("cpu")


# ============================================================
# 14. CUDA MEMORY
# ============================================================

def print_cuda_memory():

    if not torch.cuda.is_available():
        return

    allocated = (
        torch.cuda.memory_allocated() /
        1024 ** 3
    )

    reserved = (
        torch.cuda.memory_reserved() /
        1024 ** 3
    )

    peak = (
        torch.cuda.max_memory_allocated() /
        1024 ** 3
    )

    print(
        f"CUDA memory | "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB | "
        f"peak={peak:.2f} GB"
    )


# ============================================================
# 15. LOAD JSONL
# ============================================================

def load_jsonl(path):

    samples = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            samples.append(
                json.loads(line)
            )

    return samples


# ============================================================
# 16. LABEL MAPPING
# ============================================================

def get_label_mapping(dataset):

    if dataset == "DRWeibo":

        return (
            {
                "0": 0,
                "1": 1,
            },
            {
                0: "0",
                1: "1",
            },
        )

    if dataset == "PHEME":

        return (
            {
                "false": 0,
                "true": 1,
                "unverified": 2,
            },
            {
                0: "false",
                1: "true",
                2: "unverified",
            },
        )

    raise ValueError(
        f"Unsupported dataset: {dataset}"
    )


# ============================================================
# 17. PATHS
# ============================================================

def get_paths():

    if DATASET == "DRWeibo":

        data_dir = SPLIT_DIR / "DRWeibo"
        model_dir = DRWEIBO_MODEL_DIR

    elif DATASET == "PHEME":

        data_dir = (
            SPLIT_DIR /
            "PHEME" /
            f"fold_{PHEME_FOLD}"
        )

        model_dir = PHEME_MODEL_DIR

    else:

        raise ValueError(DATASET)

    return (
        data_dir / "train.jsonl",
        data_dir / "val.jsonl",
        data_dir / "test.jsonl",
        model_dir,
    )


# ============================================================
# 18. TOKENIZER
# ============================================================

def load_tokenizer(dataset, model_dir):

    if dataset == "DRWeibo":

        return BertTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )

    if dataset == "PHEME":

        return RobertaTokenizer.from_pretrained(
            str(model_dir),
            local_files_only=True
        )

    raise ValueError(dataset)


# ============================================================
# 19. GRAPH DEPTH
# ============================================================

def compute_max_depth(
    nodes,
    edges,
    root_id
):

    node_ids = {
        str(node["node_id"])
        for node in nodes
    }

    children = {
        node_id: []
        for node_id in node_ids
    }

    for edge in edges:

        if (
            not isinstance(edge, (list, tuple))
            or
            len(edge) != 2
        ):
            continue

        parent = str(edge[0])
        child = str(edge[1])

        if (
            parent in node_ids
            and
            child in node_ids
        ):
            children[parent].append(child)

    root_id = str(root_id)

    if root_id not in node_ids:
        return 0

    queue = deque(
        [(root_id, 0)]
    )

    visited = set()
    max_depth = 0

    while queue:

        node_id, depth = queue.popleft()

        if node_id in visited:
            continue

        visited.add(node_id)

        max_depth = max(
            max_depth,
            depth
        )

        for child in children.get(
            node_id,
            []
        ):
            queue.append(
                (child, depth + 1)
            )

    return max_depth


# ============================================================
# 20. GRAPH STATISTICS / SSD INPUT
# ============================================================

def extract_graph_statistics(sample):

    nodes = sample.get(
        "nodes",
        []
    )

    edges = sample.get(
        "edges",
        []
    )

    n = len(nodes)
    e = len(edges)

    existing = sample.get(
        "graph_statistics",
        {}
    )

    depth = existing.get(
        "max_depth",
        existing.get(
            "depth",
            None
        )
    )

    if depth is None:

        root_id = sample.get(
            "root_id",
            (
                nodes[0]["node_id"]
                if nodes
                else ""
            )
        )

        depth = compute_max_depth(
            nodes,
            edges,
            root_id
        )

    # Directed graph density.
    if n > 1:
        density = (
            e /
            (
                n *
                (n - 1)
            )
        )
    else:
        density = 0.0

    # Average degree.
    if n > 0:
        avg_degree = (
            2.0 *
            e /
            n
        )
    else:
        avg_degree = 0.0

    # Same working definition used in the previous SSEE/full run.
    branching = (
        e /
        max(
            float(depth),
            1.0
        )
    )

    # Fixed numerical scaling.
    stats = np.array(
        [
            np.log1p(n),
            np.log1p(e),
            np.log1p(float(depth)),
            float(density),
            np.log1p(avg_degree),
            np.log1p(branching),
        ],
        dtype=np.float32
    )

    return stats


# ============================================================
# 21. DATASET
# ============================================================

class ConversationGraphDataset(
    Dataset
):

    def __init__(
        self,
        samples,
        label2id
    ):

        self.samples = samples
        self.label2id = label2id


    def __len__(self):

        return len(
            self.samples
        )


    def __getitem__(
        self,
        idx
    ):

        sample = self.samples[idx]

        nodes = sample.get(
            "nodes",
            []
        )

        edges = sample.get(
            "edges",
            []
        )

        node_id_to_idx = {}
        texts = []

        for node_idx, node in enumerate(
            nodes
        ):

            node_id = str(
                node["node_id"]
            )

            node_id_to_idx[
                node_id
            ] = node_idx

            text = str(
                node.get(
                    "text",
                    ""
                )
            ).strip()

            if not text:
                text = "[EMPTY]"

            texts.append(text)

        indexed_edges = []

        for edge in edges:

            if (
                not isinstance(edge, (list, tuple))
                or
                len(edge) != 2
            ):
                continue

            parent = str(edge[0])
            child = str(edge[1])

            if (
                parent in node_id_to_idx
                and
                child in node_id_to_idx
            ):

                indexed_edges.append(
                    (
                        node_id_to_idx[parent],
                        node_id_to_idx[child],
                    )
                )

        label = self.label2id[
            str(
                sample["label"]
            )
        ]

        graph_stats = (
            extract_graph_statistics(
                sample
            )
        )

        return {
            "id": str(sample["id"]),
            "texts": texts,
            "edges": indexed_edges,
            "label": label,
            "graph_stats": graph_stats,
        }


# ============================================================
# 22. COLLATOR
# ============================================================

class GraphConversationCollator:

    def __init__(
        self,
        tokenizer,
        max_length
    ):

        self.tokenizer = tokenizer
        self.max_length = max_length


    def __call__(
        self,
        batch
    ):

        all_texts = []
        all_edges = []
        conversation_ids = []
        labels = []
        sample_ids = []
        graph_stats = []

        node_offset = 0

        for conv_idx, item in enumerate(
            batch
        ):

            texts = item["texts"]
            edges = item["edges"]

            num_nodes = len(texts)

            if num_nodes == 0:

                raise ValueError(
                    f"Conversation "
                    f"{item['id']} "
                    f"contains zero nodes."
                )

            all_texts.extend(texts)

            conversation_ids.extend(
                [conv_idx] *
                num_nodes
            )

            for src, dst in edges:

                all_edges.append(
                    (
                        src + node_offset,
                        dst + node_offset,
                    )
                )

            node_offset += num_nodes

            labels.append(
                item["label"]
            )

            sample_ids.append(
                item["id"]
            )

            graph_stats.append(
                item["graph_stats"]
            )

        encoded = self.tokenizer(
            all_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        if all_edges:

            edge_index = (
                torch.tensor(
                    all_edges,
                    dtype=torch.long
                )
                .t()
                .contiguous()
            )

        else:

            edge_index = torch.empty(
                (2, 0),
                dtype=torch.long
            )

        return {
            "input_ids":
                encoded["input_ids"],

            "attention_mask":
                encoded["attention_mask"],

            "edge_index":
                edge_index,

            "conversation_ids":
                torch.tensor(
                    conversation_ids,
                    dtype=torch.long
                ),

            "graph_stats":
                torch.tensor(
                    np.stack(
                        graph_stats
                    ),
                    dtype=torch.float32
                ),

            "labels":
                torch.tensor(
                    labels,
                    dtype=torch.long
                ),

            "sample_ids":
                sample_ids,
        }


# ============================================================
# 23. GRAPH EDGE PREPARATION
# ============================================================

def prepare_edge_index(
    edge_index,
    num_nodes,
    make_bidirectional=True
):

    edges = edge_index
    device = edge_index.device

    if (
        make_bidirectional
        and
        edges.size(1) > 0
    ):

        reverse = torch.stack(
            [
                edges[1],
                edges[0],
            ],
            dim=0
        )

        edges = torch.cat(
            [
                edges,
                reverse,
            ],
            dim=1
        )

    node_idx = torch.arange(
        num_nodes,
        dtype=torch.long,
        device=device
    )

    self_loops = torch.stack(
        [
            node_idx,
            node_idx,
        ],
        dim=0
    )

    edges = torch.cat(
        [
            edges,
            self_loops,
        ],
        dim=1
    )

    edge_pairs = (
        edges
        .t()
        .contiguous()
    )

    edge_pairs = torch.unique(
        edge_pairs,
        dim=0
    )

    return (
        edge_pairs
        .t()
        .contiguous()
    )


# ============================================================
# 24. MULTI-HEAD EDGE SOFTMAX
# ============================================================

def edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()

    num_heads = (
        scores_fp32.size(1)
    )

    expanded_dst = (
        dst
        .unsqueeze(1)
        .expand(
            -1,
            num_heads
        )
    )

    max_per_node = torch.full(
        (
            num_nodes,
            num_heads
        ),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )

    max_per_node.scatter_reduce_(
        0,
        expanded_dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )

    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )

    exp_scores = torch.exp(
        stabilized
    )

    denominator = torch.zeros(
        (
            num_nodes,
            num_heads
        ),
        dtype=torch.float32,
        device=scores.device
    )

    denominator.index_add_(
        0,
        dst,
        exp_scores
    )

    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================
# 25. SCALAR EDGE SOFTMAX
# ============================================================

def scalar_edge_softmax(
    scores,
    dst,
    num_nodes
):

    scores_fp32 = scores.float()

    max_per_node = torch.full(
        (num_nodes,),
        -float("inf"),
        dtype=torch.float32,
        device=scores.device
    )

    max_per_node.scatter_reduce_(
        0,
        dst,
        scores_fp32,
        reduce="amax",
        include_self=True
    )

    stabilized = (
        scores_fp32
        -
        max_per_node[dst]
    )

    exp_scores = torch.exp(
        stabilized
    )

    denominator = torch.zeros(
        (num_nodes,),
        dtype=torch.float32,
        device=scores.device
    )

    denominator.index_add_(
        0,
        dst,
        exp_scores
    )

    return (
        exp_scores /
        (
            denominator[dst]
            +
            1e-12
        )
    )


# ============================================================
# 26. VANILLA GAT LAYER
# ============================================================

class VanillaGATLayer(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim,
        num_heads,
        dropout,
        attention_dropout,
        negative_slope=0.2
    ):

        super().__init__()

        assert (
            output_dim %
            num_heads ==
            0
        )

        self.output_dim = output_dim
        self.num_heads = num_heads

        self.head_dim = (
            output_dim //
            num_heads
        )

        self.negative_slope = negative_slope

        self.linear = nn.Linear(
            input_dim,
            output_dim,
            bias=False
        )

        self.att_src = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )

        self.att_dst = nn.Parameter(
            torch.empty(
                num_heads,
                self.head_dim
            )
        )

        self.bias = nn.Parameter(
            torch.zeros(
                output_dim
            )
        )

        self.feature_dropout = nn.Dropout(
            dropout
        )

        self.attention_dropout = nn.Dropout(
            attention_dropout
        )

        self.reset_parameters()


    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.linear.weight
        )

        nn.init.xavier_uniform_(
            self.att_src
        )

        nn.init.xavier_uniform_(
            self.att_dst
        )

        nn.init.zeros_(
            self.bias
        )


    def forward(
        self,
        x,
        edge_index
    ):

        num_nodes = x.size(0)

        x = self.feature_dropout(x)

        h = self.linear(x)

        h = h.view(
            num_nodes,
            self.num_heads,
            self.head_dim
        )

        src = edge_index[0]
        dst = edge_index[1]

        src_score = (
            (
                h[src]
                *
                self.att_src
            )
            .sum(dim=-1)
        )

        dst_score = (
            (
                h[dst]
                *
                self.att_dst
            )
            .sum(dim=-1)
        )

        scores = F.leaky_relu(
            src_score + dst_score,
            negative_slope=
                self.negative_slope
        )

        alpha = edge_softmax(
            scores,
            dst,
            num_nodes
        )

        alpha = self.attention_dropout(
            alpha
        )

        # AMP dtype alignment
        alpha = alpha.to(
            dtype=h.dtype
        )

        messages = (
            h[src]
            *
            alpha.unsqueeze(-1)
        )

        output = torch.zeros(
            (
                num_nodes,
                self.num_heads,
                self.head_dim
            ),
            dtype=h.dtype,
            device=h.device
        )

        output.index_add_(
            0,
            dst,
            messages
        )

        output = output.reshape(
            num_nodes,
            self.output_dim
        )

        return (
            output +
            self.bias
        )


# ============================================================
# 27. SEM
# ============================================================

class StructuralEvidenceMining(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        evidence_dim,
        dropout
    ):

        super().__init__()

        self.query_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )

        self.key_proj = nn.Linear(
            hidden_dim,
            evidence_dim,
            bias=False
        )

        self.evidence_vector = nn.Parameter(
            torch.empty(
                evidence_dim
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.reset_parameters()


    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.query_proj.weight
        )

        nn.init.xavier_uniform_(
            self.key_proj.weight
        )

        nn.init.normal_(
            self.evidence_vector,
            mean=0.0,
            std=0.02
        )


    def forward(
        self,
        h_g,
        edge_index
    ):

        num_nodes = h_g.size(0)

        src = edge_index[0]
        dst = edge_index[1]

        q_i = self.query_proj(
            h_g[dst]
        )

        k_j = self.key_proj(
            h_g[src]
        )

        evidence_hidden = torch.tanh(
            q_i + k_j
        )

        evidence_scores = (
            evidence_hidden
            *
            self.evidence_vector
        ).sum(dim=-1)

        alpha = scalar_edge_softmax(
            evidence_scores,
            dst,
            num_nodes
        )

        alpha = self.dropout(
            alpha
        )

        alpha = alpha.to(
            dtype=h_g.dtype
        )

        messages = (
            h_g[src]
            *
            alpha.unsqueeze(-1)
        )

        h_m = torch.zeros(
            h_g.shape,
            dtype=h_g.dtype,
            device=h_g.device
        )

        h_m.index_add_(
            0,
            dst,
            messages
        )

        return h_m


# ============================================================
# 28. AGCC
# ============================================================

class AdaptiveGlobalContextCompensation(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim
    ):

        super().__init__()

        self.gate = nn.Linear(
            hidden_dim * 2 +
            stat_dim,
            hidden_dim
        )


    def forward(
        self,
        h_m,
        conversation_ids,
        graph_stats,
        batch_size
    ):

        hidden_dim = h_m.size(-1)

        global_context = torch.zeros(
            (
                batch_size,
                hidden_dim
            ),
            dtype=h_m.dtype,
            device=h_m.device
        )

        global_context.index_add_(
            0,
            conversation_ids,
            h_m
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=h_m.dtype,
                device=h_m.device
            )
        )

        global_context = (
            global_context /
            counts
        )

        node_global = (
            global_context[
                conversation_ids
            ]
        )

        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_m.dtype
            )
        )

        gate_input = torch.cat(
            [
                h_m,
                node_global,
                node_stats,
            ],
            dim=-1
        )

        gamma = torch.sigmoid(
            self.gate(
                gate_input
            )
        )

        h_c = (
            h_m +
            gamma *
            node_global
        )

        return h_c


# ============================================================
# 29. ASE
# ============================================================

class AdaptiveStructuralEnhancement(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        stat_dim,
        dropout
    ):

        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim +
                stat_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
        )


    def forward(
        self,
        h_c,
        h_g,
        conversation_ids,
        graph_stats
    ):

        node_stats = (
            graph_stats[
                conversation_ids
            ]
            .to(
                dtype=h_c.dtype
            )
        )

        ase_input = torch.cat(
            [
                h_c,
                node_stats,
            ],
            dim=-1
        )

        correction = self.mlp(
            ase_input
        )

        # Residual to initial GAT structural representation.
        h_e = (
            h_g +
            correction
        )

        return h_e



# ============================================================
# 30. CONTROLLED MODEL VARIANTS
# ============================================================

class SemanticOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        h_graph = self.conversation_mean_pool(
            h_x,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


class GATOnlyClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        encoder_dim = self.encoder.config.hidden_size

        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        h_graph = self.conversation_mean_pool(
            h_g,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


class GATSSEEClassifier(nn.Module):

    def __init__(self, dataset, model_dir, num_classes):
        super().__init__()

        if dataset == "DRWeibo":
            self.encoder = BertModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        elif dataset == "PHEME":
            self.encoder = RobertaModel.from_pretrained(
                str(model_dir),
                local_files_only=True
            )
        else:
            raise ValueError(dataset)

        self.encoder.gradient_checkpointing_enable()

        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False

        encoder_dim = self.encoder.config.hidden_size

        self.gat1 = VanillaGATLayer(
            input_dim=encoder_dim,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.gat2 = VanillaGATLayer(
            input_dim=GAT_HIDDEN_DIM,
            output_dim=GAT_HIDDEN_DIM,
            num_heads=GAT_HEADS,
            dropout=GAT_DROPOUT,
            attention_dropout=GAT_ATTENTION_DROPOUT,
            negative_slope=LEAKY_RELU_SLOPE
        )

        self.sem = StructuralEvidenceMining(
            hidden_dim=GAT_HIDDEN_DIM,
            evidence_dim=SEM_EVIDENCE_DIM,
            dropout=SSEE_DROPOUT
        )

        self.agcc = AdaptiveGlobalContextCompensation(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM
        )

        self.ase = AdaptiveStructuralEnhancement(
            hidden_dim=GAT_HIDDEN_DIM,
            stat_dim=GRAPH_STAT_DIM,
            dropout=SSEE_DROPOUT
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.classifier = nn.Linear(
            GAT_HIDDEN_DIM,
            num_classes
        )

    def encode_nodes(self, input_ids, attention_mask):
        chunks = []
        total_nodes = input_ids.size(0)

        for start in range(0, total_nodes, NODE_CHUNK_SIZE):
            end = min(start + NODE_CHUNK_SIZE, total_nodes)

            outputs = self.encoder(
                input_ids=input_ids[start:end],
                attention_mask=attention_mask[start:end]
            )

            chunks.append(
                outputs.last_hidden_state[:, 0, :]
            )

        return torch.cat(chunks, dim=0)

    @staticmethod
    def conversation_mean_pool(
        node_features,
        conversation_ids,
        batch_size
    ):
        hidden_dim = node_features.size(-1)

        pooled = torch.zeros(
            (batch_size, hidden_dim),
            dtype=node_features.dtype,
            device=node_features.device
        )

        pooled.index_add_(
            0,
            conversation_ids,
            node_features
        )

        counts = (
            torch.bincount(
                conversation_ids,
                minlength=batch_size
            )
            .clamp(min=1)
            .unsqueeze(1)
            .to(
                dtype=node_features.dtype,
                device=node_features.device
            )
        )

        return pooled / counts

    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):
        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        h_m = self.sem(
            h_g,
            graph_edge_index
        )

        h_c = self.agcc(
            h_m,
            conversation_ids,
            graph_stats,
            batch_size
        )

        h_e = self.ase(
            h_c,
            h_g,
            conversation_ids,
            graph_stats
        )

        h_graph = self.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


# ============================================================
# 30B. SSEE REMOVE-ONE ABLATION MODEL
# ============================================================

class SSEEAblationClassifier(GATSSEEClassifier):

    """
    Controlled remove-one ablations of SSEE.

    full SSEE:
        H^G -> SEM -> AGCC -> ASE -> pooling

    wo_sem:
        H^G --------> AGCC -> ASE -> pooling

    wo_agcc:
        H^G -> SEM --------> ASE -> pooling

    wo_ase:
        H^G -> SEM -> AGCC -----> pooling

    No new module is introduced in any ablation.
    """

    def __init__(
        self,
        dataset,
        model_dir,
        num_classes,
        ablation
    ):

        super().__init__(
            dataset,
            model_dir,
            num_classes
        )

        valid = {
            "wo_sem",
            "wo_agcc",
            "wo_ase",
        }

        if ablation not in valid:
            raise ValueError(
                f"Unsupported ablation: {ablation}"
            )

        self.ablation = ablation


    def forward(
        self,
        input_ids,
        attention_mask,
        edge_index,
        conversation_ids,
        graph_stats,
        batch_size
    ):

        h_x = self.encode_nodes(
            input_ids,
            attention_mask
        )

        num_nodes = h_x.size(0)

        graph_edge_index = prepare_edge_index(
            edge_index,
            num_nodes,
            MAKE_BIDIRECTIONAL
        )

        h_g = F.elu(
            self.gat1(
                h_x,
                graph_edge_index
            )
        )

        h_g = F.elu(
            self.gat2(
                h_g,
                graph_edge_index
            )
        )

        # ----------------------------------------------------
        # SEM
        # ----------------------------------------------------
        if self.ablation == "wo_sem":
            h_m = h_g
        else:
            h_m = self.sem(
                h_g,
                graph_edge_index
            )

        # ----------------------------------------------------
        # AGCC
        # ----------------------------------------------------
        if self.ablation == "wo_agcc":
            h_c = h_m
        else:
            h_c = self.agcc(
                h_m,
                conversation_ids,
                graph_stats,
                batch_size
            )

        # ----------------------------------------------------
        # ASE
        # ----------------------------------------------------
        if self.ablation == "wo_ase":
            h_e = h_c
        else:
            h_e = self.ase(
                h_c,
                h_g,
                conversation_ids,
                graph_stats
            )

        h_graph = self.conversation_mean_pool(
            h_e,
            conversation_ids,
            batch_size
        )

        return self.classifier(
            self.dropout(h_graph)
        )


# ============================================================
# 31. EARLY-WINDOW EXPERIMENT CONFIG
# ============================================================

WINDOWS = [10, 30, 60, 120, 240]

MODEL_VARIANTS = [
    "semantic",
    "gat",
    "ssee",
]

# To test SSEE only first:
# MODEL_VARIANTS = ["ssee"]

EARLY_DATA_DIR = (
    BASE_DIR /
    "DRWeibo"
)

EARLY_RESULTS_DIR = (
    BASE_DIR /
    "experiment_results" /
    "drweibo_early_windows"
)


# ============================================================
# 32. EARLY-WINDOW FILE
# ============================================================

def get_early_window_path(window):

    candidates = [
        EARLY_DATA_DIR /
        f"drweibo_{window}min.jsonl",

        BASE_DIR /
        f"drweibo_{window}min.jsonl",

        BASE_DIR /
        "DRWeibo" /
        f"DRWeibo_{window}min.jsonl",

        BASE_DIR /
        "DRWeibo" /
        f"drweibo_{window}_min.jsonl",
    ]

    for path in candidates:

        if path.exists():
            return path

    searched = "\n".join(
        str(path)
        for path in candidates
    )

    raise FileNotFoundError(
        f"Cannot find {window}-min DRWeibo file.\n"
        f"Searched:\n{searched}"
    )


# ============================================================
# 33. LOCKED SPLIT IDS
# ============================================================

def load_fixed_split_ids():

    split_dir = (
        SPLIT_DIR /
        "DRWeibo"
    )

    split_ids = {}

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        path = (
            split_dir /
            f"{split_name}.jsonl"
        )

        samples = load_jsonl(
            path
        )

        ids = [
            str(sample["id"])
            for sample in samples
        ]

        if len(ids) != len(set(ids)):

            raise ValueError(
                f"Duplicate IDs found in {path}"
            )

        split_ids[
            split_name
        ] = ids

    all_ids = (
        split_ids["train"]
        +
        split_ids["val"]
        +
        split_ids["test"]
    )

    if len(all_ids) != len(set(all_ids)):

        raise ValueError(
            "Train/val/test ID overlap detected."
        )

    return split_ids


# ============================================================
# 34. APPLY LOCKED SPLIT TO EARLY WINDOW
# ============================================================

def build_window_splits(
    window_path,
    split_ids
):

    samples = load_jsonl(
        window_path
    )

    sample_map = {}

    for sample in samples:

        sample_id = str(
            sample["id"]
        )

        if sample_id in sample_map:

            raise ValueError(
                f"Duplicate ID in "
                f"{window_path}: "
                f"{sample_id}"
            )

        sample_map[
            sample_id
        ] = sample

    window_splits = {}

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        wanted_ids = (
            split_ids[
                split_name
            ]
        )

        missing = [
            sample_id
            for sample_id in wanted_ids
            if sample_id not in sample_map
        ]

        if missing:

            raise ValueError(
                f"{window_path.name}: "
                f"{len(missing)} IDs missing from "
                f"{split_name}. Examples: "
                f"{missing[:10]}"
            )

        window_splits[
            split_name
        ] = [
            sample_map[
                sample_id
            ]
            for sample_id in wanted_ids
        ]

    return window_splits


# ============================================================
# 35. METRICS
# ============================================================

def calculate_metrics(
    labels,
    predictions,
    id2label
):

    label_ids = list(
        id2label.keys()
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average="macro",
        zero_division=0
    )

    class_f1 = f1_score(
        labels,
        predictions,
        labels=label_ids,
        average=None,
        zero_division=0
    )

    report = classification_report(
        labels,
        predictions,
        labels=label_ids,
        target_names=[
            id2label[i]
            for i in label_ids
        ],
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "class_f1": class_f1,
        "report": report,
    }


# ============================================================
# 36. EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    dataloader,
    device,
    criterion,
    id2label,
    use_amp
):

    model.eval()

    total_loss = 0.0

    all_labels = []
    all_predictions = []
    all_sample_ids = []

    for batch in tqdm(
        dataloader,
        desc="Evaluating",
        leave=False
    ):

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )

        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )

        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        batch_size = labels.size(0)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                edge_index=edge_index,
                conversation_ids=conversation_ids,
                graph_stats=graph_stats,
                batch_size=batch_size
            )

            loss = criterion(
                logits,
                labels
            )

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.cpu().tolist()
        )

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_sample_ids.extend(
            batch["sample_ids"]
        )

    metrics = calculate_metrics(
        all_labels,
        all_predictions,
        id2label
    )

    return {
        "loss":
            total_loss /
            max(len(dataloader), 1),

        **metrics,

        "labels":
            all_labels,

        "predictions":
            all_predictions,

        "sample_ids":
            all_sample_ids,
    }


# ============================================================
# 37. OPTIMIZER STEP
# ============================================================

def optimizer_update(
    model,
    optimizer,
    scaler,
    use_amp
):

    if use_amp:

        scaler.unscale_(
            optimizer
        )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRADIENT_CLIP
    )

    if use_amp:

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        optimizer.step()

    optimizer.zero_grad(
        set_to_none=True
    )


# ============================================================
# 38. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device,
    scaler,
    use_amp
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss = 0.0
    num_batches = len(dataloader)

    progress = tqdm(
        enumerate(
            dataloader,
            start=1
        ),
        total=num_batches,
        desc="Training"
    )

    for step, batch in progress:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        edge_index = batch["edge_index"].to(
            device,
            non_blocking=True
        )

        conversation_ids = batch["conversation_ids"].to(
            device,
            non_blocking=True
        )

        graph_stats = batch["graph_stats"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        batch_size = labels.size(0)

        try:

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    edge_index=edge_index,
                    conversation_ids=conversation_ids,
                    graph_stats=graph_stats,
                    batch_size=batch_size
                )

                raw_loss = criterion(
                    logits,
                    labels
                )

                loss = (
                    raw_loss /
                    GRAD_ACCUM_STEPS
                )

            if use_amp:

                scaler.scale(
                    loss
                ).backward()

            else:

                loss.backward()

            total_loss += raw_loss.item()

            should_update = (
                step % GRAD_ACCUM_STEPS == 0
                or
                step == num_batches
            )

            if should_update:

                optimizer_update(
                    model,
                    optimizer,
                    scaler,
                    use_amp
                )

            progress.set_postfix(
                loss=f"{raw_loss.item():.4f}",
                nodes=input_ids.size(0),
                edges=edge_index.size(1)
            )

        except torch.OutOfMemoryError:

            print("\nCUDA OOM")
            print("Nodes:", input_ids.size(0))
            print("Edges:", edge_index.size(1))

            print_cuda_memory()

            optimizer.zero_grad(
                set_to_none=True
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise

    return (
        total_loss /
        max(num_batches, 1)
    )


# ============================================================
# 39. BUILD MODEL
# ============================================================

def build_model(
    variant,
    model_dir,
    num_classes
):

    if variant == "semantic":

        return SemanticOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant == "gat":

        return GATOnlyClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant == "ssee":

        return GATSSEEClassifier(
            DATASET,
            model_dir,
            num_classes
        )

    if variant in {
        "wo_sem",
        "wo_agcc",
        "wo_ase",
    }:

        return SSEEAblationClassifier(
            DATASET,
            model_dir,
            num_classes,
            ablation=variant
        )

    raise ValueError(
        f"Unknown model variant: {variant}"
    )


# ============================================================
# 40. CONTROLLED OPTIMIZER
# ============================================================

def build_optimizer(model):

    encoder_parameters = list(
        model.encoder.parameters()
    )

    encoder_ids = {
        id(parameter)
        for parameter in encoder_parameters
    }

    new_parameters = [
        parameter
        for parameter in model.parameters()
        if id(parameter)
        not in encoder_ids
    ]

    return torch.optim.AdamW(
        [
            {
                "params":
                    encoder_parameters,
                "lr":
                    ENCODER_LR
            },
            {
                "params":
                    new_parameters,
                "lr":
                    NEW_MODULE_LR
            },
        ],
        weight_decay=WEIGHT_DECAY
    )


# ============================================================
# 41. SAVE PREDICTIONS
# ============================================================

def save_predictions(
    sample_ids,
    labels,
    predictions,
    id2label,
    path
):

    rows = []

    for sample_id, y_true, y_pred in zip(
        sample_ids,
        labels,
        predictions
    ):

        rows.append(
            {
                "sample_id":
                    sample_id,
                "true_id":
                    y_true,
                "true_label":
                    id2label[y_true],
                "pred_id":
                    y_pred,
                "pred_label":
                    id2label[y_pred],
            }
        )

    pd.DataFrame(
        rows
    ).to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 42. ONE MODEL × ONE WINDOW


# ============================================================
# 42. PHEME 9-FOLD LEAVE-ONE-EVENT-OUT MULTI-SEED


# ============================================================
# 42. DRWEIBO 10MIN SSEE PARAMETER SENSITIVITY


# ============================================================
# 42. COMPLEXITY / EFFICIENCY BENCHMARK
#     DRWeibo 10min | Semantic vs GAT vs SSEE
#     RTX 4090 auto-calibration + resumable per-repeat results
# ============================================================

import math
from collections import defaultdict


RESULT_DIR = (
    BASE_DIR /
    "experiment_results" /
    "complexity_analysis_drweibo_10min_v3"
)

WINDOW_MIN = 10
SEED = 42

MODELS = [
    "semantic",
    "gat",
    "ssee",
]

# Benchmark design.
# Same batch size is used for all 3 models after calibration on SSEE.
TARGET_PEAK_LOW_GIB = 16.0
TARGET_PEAK_HIGH_GIB = 19.0

BATCH_CANDIDATES = [64, 48, 32, 24, 16, 12, 8, 4]

NODE_CHUNK_CANDIDATES = [128, 96, 64]

WARMUP_BATCHES = 5
INFERENCE_BATCHES = 30
TRAIN_BATCHES = 20
REPEATS = 3

# We benchmark with AMP + TF32 because this matches the actual GPU execution
# environment used in the experiments.
USE_TF32 = True

# For complexity benchmarking we turn encoder gradient checkpointing OFF:
# inference does not benefit from it, and training runtime/memory should reflect
# the direct computational cost. The batch calibration protects against OOM.
BENCHMARK_GRADIENT_CHECKPOINTING = True


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def is_cuda_oom(exc):
    s = str(exc).lower()
    return (
        isinstance(exc, torch.cuda.OutOfMemoryError)
        or
        "cuda out of memory" in s
    )


def load_split_ids(which):
    p = (
        BASE_DIR /
        "splits" /
        "DRWeibo" /
        f"{which}.jsonl"
    )

    ids = []

    with p.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                ids.append(
                    str(json.loads(line)["id"])
                )

    return ids


def load_window_samples():
    p = (
        BASE_DIR /
        "DRWeibo" /
        f"drweibo_{WINDOW_MIN}min.jsonl"
    )

    rows = load_jsonl(p)
    by_id = {
        str(x["id"]): x
        for x in rows
    }

    out = {}

    for split in ["train", "val", "test"]:
        ids = load_split_ids(split)

        missing = [
            x for x in ids
            if x not in by_id
        ]

        if missing:
            raise RuntimeError(
                f"{split}: {len(missing)} IDs missing "
                f"from {p.name}. Examples={missing[:5]}"
            )

        out[split] = [
            by_id[x]
            for x in ids
        ]

    return out


def count_nodes(sample):
    return len(sample.get("nodes", []))


def make_loader(
    samples,
    tokenizer,
    label2id,
    batch_size,
    shuffle=False,
):
    collator = GraphConversationCollator(
        tokenizer,
        MAX_LENGTH
    )

    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )

    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = 6

    return DataLoader(
        ConversationGraphDataset(
            samples,
            label2id
        ),
        **kwargs
    )


def configure_encoder_checkpointing(model):
    enc = getattr(model, "encoder", None)

    if enc is None:
        return

    if BENCHMARK_GRADIENT_CHECKPOINTING:
        if hasattr(enc, "gradient_checkpointing_enable"):
            enc.gradient_checkpointing_enable()
    else:
        if hasattr(enc, "gradient_checkpointing_disable"):
            enc.gradient_checkpointing_disable()


def count_parameters(model):
    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


def get_cuda_peak_gib():
    if not torch.cuda.is_available():
        return None

    return (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )


def reset_cuda_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def build_benchmark_model(
    variant,
    device,
    num_classes,
):
    model = build_model(
        variant,
        DRWEIBO_MODEL_DIR,
        num_classes
    ).to(device)

    configure_encoder_checkpointing(model)

    return model


def one_training_step(
    model,
    batch,
    optimizer,
    criterion,
    scaler,
    device,
    use_amp,
):
    optimizer.zero_grad(
        set_to_none=True
    )

    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch["attention_mask"].to(
        device,
        non_blocking=True
    )

    edge_index = batch["edge_index"].to(
        device,
        non_blocking=True
    )

    conversation_ids = batch["conversation_ids"].to(
        device,
        non_blocking=True
    )

    graph_stats = batch["graph_stats"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    batch_size = labels.size(0)

    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp
    ):
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            edge_index=edge_index,
            conversation_ids=conversation_ids,
            graph_stats=graph_stats,
            batch_size=batch_size
        )

        loss = criterion(
            logits,
            labels
        )

    if use_amp:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    return float(
        loss.detach().item()
    )


@torch.no_grad()
def one_inference_step(
    model,
    batch,
    device,
    use_amp,
):
    input_ids = batch["input_ids"].to(
        device,
        non_blocking=True
    )

    attention_mask = batch["attention_mask"].to(
        device,
        non_blocking=True
    )

    edge_index = batch["edge_index"].to(
        device,
        non_blocking=True
    )

    conversation_ids = batch["conversation_ids"].to(
        device,
        non_blocking=True
    )

    graph_stats = batch["graph_stats"].to(
        device,
        non_blocking=True
    )

    labels = batch["labels"].to(
        device,
        non_blocking=True
    )

    batch_size = labels.size(0)

    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=use_amp
    ):
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            edge_index=edge_index,
            conversation_ids=conversation_ids,
            graph_stats=graph_stats,
            batch_size=batch_size
        )

    return logits


def calibrate_common_batch(
    split_samples,
    tokenizer,
    label2id,
    device,
    use_amp,
):
    """
    Calibrate on SSEE using the longest 10-minute conversations.
    The selected batch/chunk pair is then used unchanged for all models.
    """

    global BATCH_SIZE
    global NODE_CHUNK_SIZE

    print("\n" + "=" * 96)
    print("RTX 4090 COMMON-BATCH CALIBRATION ON SSEE")
    print("=" * 96)

    # Stress subset: longest conversations first.
    stress_samples = sorted(
        split_samples["train"],
        key=count_nodes,
        reverse=True,
    )[:256]

    print(
        f"Stress subset: {len(stress_samples)} longest "
        f"DRWeibo {WINDOW_MIN}min conversations"
    )

    trial_rows = []

    for chunk in NODE_CHUNK_CANDIDATES:

        for batch_size in BATCH_CANDIDATES:

            BATCH_SIZE = batch_size
            NODE_CHUNK_SIZE = chunk

            cleanup_cuda()

            model = None
            optimizer = None
            scaler = None

            try:
                loader = make_loader(
                    stress_samples,
                    tokenizer,
                    label2id,
                    batch_size,
                    shuffle=False,
                )

                model = build_benchmark_model(
                    "ssee",
                    device,
                    len(label2id)
                )

                optimizer = build_optimizer(
                    model
                )

                criterion = nn.CrossEntropyLoss()

                scaler = torch.amp.GradScaler(
                    "cuda",
                    enabled=use_amp
                )

                model.train()

                reset_cuda_peak()

                # One full forward+backward step on the longest conversations is
                # sufficient for memory calibration and avoids repeatedly stressing VRAM.
                iterator = iter(loader)

                steps = min(
                    1,
                    len(loader)
                )

                for _ in range(steps):
                    batch = next(iterator)

                    one_training_step(
                        model,
                        batch,
                        optimizer,
                        criterion,
                        scaler,
                        device,
                        use_amp,
                    )

                if torch.cuda.is_available():
                    torch.cuda.synchronize()

                peak = get_cuda_peak_gib()

                trial_rows.append({
                    "batch_size": batch_size,
                    "node_chunk_size": chunk,
                    "peak_cuda_gib": peak,
                    "status": "PASS",
                })

                pd.DataFrame(trial_rows).to_csv(
                    RESULT_DIR / "batch_calibration.csv",
                    index=False,
                    encoding="utf-8-sig"
                )

                print(
                    f"PASS | batch={batch_size:<3d} "
                    f"chunk={chunk:<3d} "
                    f"peak={peak:.2f} GiB"
                )

                # Prefer a configuration inside the requested 17-20 GiB target.
                if (
                    peak is not None
                    and
                    TARGET_PEAK_LOW_GIB
                    <= peak
                    <= TARGET_PEAK_HIGH_GIB
                ):
                    pd.DataFrame(
                        trial_rows
                    ).to_csv(
                        RESULT_DIR /
                        "batch_calibration.csv",
                        index=False,
                        encoding="utf-8-sig"
                    )

                    return {
                        "batch_size": batch_size,
                        "node_chunk_size": chunk,
                        "peak_cuda_gib": peak,
                    }

                # If a very large batch still stays below target, keep it.
                # We iterate large-to-small, so the first successful max-throughput
                # candidate is a reasonable choice when the target cannot be reached.
                if (
                    peak is not None
                    and
                    peak < TARGET_PEAK_LOW_GIB
                    and
                    batch_size == BATCH_CANDIDATES[0]
                    and
                    chunk == NODE_CHUNK_CANDIDATES[0]
                ):
                    print(
                        "Largest configured candidate still below target VRAM; "
                        "using this maximum-throughput profile."
                    )

                    pd.DataFrame(
                        trial_rows
                    ).to_csv(
                        RESULT_DIR /
                        "batch_calibration.csv",
                        index=False,
                        encoding="utf-8-sig"
                    )

                    return {
                        "batch_size": batch_size,
                        "node_chunk_size": chunk,
                        "peak_cuda_gib": peak,
                    }

            except Exception as exc:
                if is_cuda_oom(exc):
                    print(
                        f"OOM  | batch={batch_size:<3d} "
                        f"chunk={chunk:<3d}"
                    )

                    trial_rows.append({
                        "batch_size": batch_size,
                        "node_chunk_size": chunk,
                        "peak_cuda_gib": None,
                        "status": "OOM",
                    })

                    pd.DataFrame(trial_rows).to_csv(
                        RESULT_DIR / "batch_calibration.csv",
                        index=False,
                        encoding="utf-8-sig"
                    )

                else:
                    raise

            finally:
                # Drop every potentially GPU-owning reference before empty_cache().
                for _name in ["batch", "iterator", "loader"]:
                    try:
                        del locals()[_name]
                    except Exception:
                        pass

                if model is not None:
                    del model

                if optimizer is not None:
                    del optimizer

                if scaler is not None:
                    del scaler

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    try:
                        torch.cuda.ipc_collect()
                    except Exception:
                        pass

    pd.DataFrame(
        trial_rows
    ).to_csv(
        RESULT_DIR /
        "batch_calibration.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Conservative final fallback.
    return {
        "batch_size": 8,
        "node_chunk_size": 64,
        "peak_cuda_gib": None,
    }


def repeat_result_path(
    variant,
    repeat_index,
):
    return (
        RESULT_DIR /
        variant /
        f"repeat_{repeat_index:02d}.csv"
    )


def load_repeat_result(
    variant,
    repeat_index,
):
    p = repeat_result_path(
        variant,
        repeat_index
    )

    if not p.exists():
        return None

    try:
        df = pd.read_csv(p)
        if len(df) == 1:
            return df.iloc[0].to_dict()
    except Exception:
        return None

    return None


def benchmark_one_repeat(
    variant,
    repeat_index,
    split_samples,
    tokenizer,
    label2id,
    device,
    use_amp,
    batch_size,
    node_chunk_size,
):
    global BATCH_SIZE
    global NODE_CHUNK_SIZE

    BATCH_SIZE = int(batch_size)
    NODE_CHUNK_SIZE = int(node_chunk_size)

    out_dir = RESULT_DIR / variant
    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\n" + "=" * 96)
    print(
        f"COMPLEXITY | {variant.upper()} | "
        f"repeat {repeat_index}/{REPEATS} | "
        f"batch={BATCH_SIZE} | chunk={NODE_CHUNK_SIZE}"
    )
    print("=" * 96)

    set_seed(
        SEED + repeat_index - 1
    )

    cleanup_cuda()

    model = build_benchmark_model(
        variant,
        device,
        len(label2id)
    )

    total_params, trainable_params = count_parameters(
        model
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = build_optimizer(
        model
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    train_loader = make_loader(
        split_samples["train"],
        tokenizer,
        label2id,
        BATCH_SIZE,
        shuffle=False,
    )

    test_loader = make_loader(
        split_samples["test"],
        tokenizer,
        label2id,
        BATCH_SIZE,
        shuffle=False,
    )

    # --------------------------------------------------------
    # Inference benchmark
    # --------------------------------------------------------
    model.eval()

    test_iter = iter(test_loader)

    for _ in range(
        min(WARMUP_BATCHES, len(test_loader))
    ):
        batch = next(test_iter)
        one_inference_step(
            model,
            batch,
            device,
            use_amp
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    reset_cuda_peak()

    inference_times = []
    inference_samples = 0

    test_iter = iter(test_loader)

    n_inf = min(
        INFERENCE_BATCHES,
        len(test_loader)
    )

    for _ in range(n_inf):
        batch = next(test_iter)

        n_batch = int(
            batch["labels"].shape[0]
        )

        if torch.cuda.is_available():
            start_evt = torch.cuda.Event(
                enable_timing=True
            )
            end_evt = torch.cuda.Event(
                enable_timing=True
            )

            start_evt.record()

            one_inference_step(
                model,
                batch,
                device,
                use_amp
            )

            end_evt.record()
            torch.cuda.synchronize()

            elapsed_ms = (
                start_evt.elapsed_time(
                    end_evt
                )
            )

        else:
            t0 = time.perf_counter()

            one_inference_step(
                model,
                batch,
                device,
                use_amp
            )

            elapsed_ms = (
                time.perf_counter() - t0
            ) * 1000.0

        inference_times.append(
            elapsed_ms
        )
        inference_samples += n_batch

    inference_peak = get_cuda_peak_gib()

    inference_total_ms = sum(
        inference_times
    )

    inference_latency_ms_per_batch = (
        inference_total_ms /
        max(len(inference_times), 1)
    )

    inference_ms_per_sample = (
        inference_total_ms /
        max(inference_samples, 1)
    )

    inference_throughput = (
        inference_samples /
        max(inference_total_ms / 1000.0, 1e-9)
    )

    # --------------------------------------------------------
    # Training-step benchmark
    # --------------------------------------------------------
    model.train()

    train_iter = iter(train_loader)

    # Warmup optimizer steps.
    for _ in range(
        min(WARMUP_BATCHES, len(train_loader))
    ):
        batch = next(train_iter)

        one_training_step(
            model,
            batch,
            optimizer,
            criterion,
            scaler,
            device,
            use_amp,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    reset_cuda_peak()

    train_times = []
    train_samples = 0

    train_iter = iter(train_loader)

    n_train = min(
        TRAIN_BATCHES,
        len(train_loader)
    )

    for _ in range(n_train):
        batch = next(train_iter)

        n_batch = int(
            batch["labels"].shape[0]
        )

        if torch.cuda.is_available():
            start_evt = torch.cuda.Event(
                enable_timing=True
            )
            end_evt = torch.cuda.Event(
                enable_timing=True
            )

            start_evt.record()

            one_training_step(
                model,
                batch,
                optimizer,
                criterion,
                scaler,
                device,
                use_amp,
            )

            end_evt.record()
            torch.cuda.synchronize()

            elapsed_ms = (
                start_evt.elapsed_time(
                    end_evt
                )
            )

        else:
            t0 = time.perf_counter()

            one_training_step(
                model,
                batch,
                optimizer,
                criterion,
                scaler,
                device,
                use_amp,
            )

            elapsed_ms = (
                time.perf_counter() - t0
            ) * 1000.0

        train_times.append(
            elapsed_ms
        )
        train_samples += n_batch

    training_peak = get_cuda_peak_gib()

    train_total_ms = sum(
        train_times
    )

    training_ms_per_batch = (
        train_total_ms /
        max(len(train_times), 1)
    )

    training_ms_per_sample = (
        train_total_ms /
        max(train_samples, 1)
    )

    training_throughput = (
        train_samples /
        max(train_total_ms / 1000.0, 1e-9)
    )

    result = {
        "dataset": "DRWeibo",
        "window_min": WINDOW_MIN,
        "model": variant,
        "repeat": repeat_index,
        "batch_size": BATCH_SIZE,
        "node_chunk_size": NODE_CHUNK_SIZE,
        "amp": bool(use_amp),
        "tf32": bool(USE_TF32),
        "gradient_checkpointing":
            bool(BENCHMARK_GRADIENT_CHECKPOINTING),
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "total_params_million":
            total_params / 1e6,
        "trainable_params_million":
            trainable_params / 1e6,
        "inference_batches":
            len(inference_times),
        "inference_samples":
            inference_samples,
        "inference_latency_ms_per_batch":
            inference_latency_ms_per_batch,
        "inference_ms_per_sample":
            inference_ms_per_sample,
        "inference_throughput_samples_per_sec":
            inference_throughput,
        "inference_peak_cuda_gib":
            inference_peak,
        "training_batches":
            len(train_times),
        "training_samples":
            train_samples,
        "training_ms_per_batch":
            training_ms_per_batch,
        "training_ms_per_sample":
            training_ms_per_sample,
        "training_throughput_samples_per_sec":
            training_throughput,
        "training_peak_cuda_gib":
            training_peak,
    }

    pd.DataFrame(
        [result]
    ).to_csv(
        repeat_result_path(
            variant,
            repeat_index
        ),
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Params={total_params/1e6:.2f}M | "
        f"Infer={inference_throughput:.2f} samples/s | "
        f"Train={training_throughput:.2f} samples/s"
    )

    if torch.cuda.is_available():
        print(
            f"Inference peak={inference_peak:.2f} GiB | "
            f"Training peak={training_peak:.2f} GiB"
        )

    del model
    del optimizer
    del scaler

    cleanup_cuda()

    return result


def aggregate_results():
    rows = []

    for variant in MODELS:
        for repeat_index in range(
            1,
            REPEATS + 1
        ):
            row = load_repeat_result(
                variant,
                repeat_index
            )

            if row is not None:
                rows.append(row)

    if not rows:
        return None

    df = pd.DataFrame(rows)

    df.to_csv(
        RESULT_DIR /
        "all_complexity_runs.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary = (
        df
        .groupby(
            "model",
            as_index=False
        )
        .agg(
            repeats=("repeat", "count"),
            total_params_million=(
                "total_params_million",
                "mean"
            ),
            trainable_params_million=(
                "trainable_params_million",
                "mean"
            ),
            inference_ms_per_sample_mean=(
                "inference_ms_per_sample",
                "mean"
            ),
            inference_ms_per_sample_std=(
                "inference_ms_per_sample",
                "std"
            ),
            inference_throughput_mean=(
                "inference_throughput_samples_per_sec",
                "mean"
            ),
            inference_throughput_std=(
                "inference_throughput_samples_per_sec",
                "std"
            ),
            inference_peak_cuda_gib_max=(
                "inference_peak_cuda_gib",
                "max"
            ),
            training_ms_per_sample_mean=(
                "training_ms_per_sample",
                "mean"
            ),
            training_ms_per_sample_std=(
                "training_ms_per_sample",
                "std"
            ),
            training_throughput_mean=(
                "training_throughput_samples_per_sec",
                "mean"
            ),
            training_throughput_std=(
                "training_throughput_samples_per_sec",
                "std"
            ),
            training_peak_cuda_gib_max=(
                "training_peak_cuda_gib",
                "max"
            ),
        )
    )

    summary.to_csv(
        RESULT_DIR /
        "complexity_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    return summary


def main():
    global BATCH_SIZE
    global NODE_CHUNK_SIZE

    print("\n" + "=" * 96)
    print("COMPLEXITY / EFFICIENCY ANALYSIS")
    print("DRWEIBO 10MIN | SEMANTIC vs GAT vs SSEE")
    print("RTX 4090 SAFE AUTO-CALIBRATION + RESUMABLE REPEATS")
    print("=" * 96)

    RESULT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = USE_TF32
        torch.backends.cudnn.allow_tf32 = USE_TF32
        try:
            torch.set_float32_matmul_precision(
                "high"
            )
        except Exception:
            pass

    label2id, id2label = get_label_mapping(
        "DRWeibo"
    )

    tokenizer = load_tokenizer(
        "DRWeibo",
        DRWEIBO_MODEL_DIR
    )

    device = get_device()
    use_amp = (
        device.type == "cuda"
    )

    split_samples = load_window_samples()

    print(
        f"Split sizes | "
        f"train={len(split_samples['train'])}, "
        f"val={len(split_samples['val'])}, "
        f"test={len(split_samples['test'])}"
    )

    # --------------------------------------------------------
    # Resume calibration if it already exists.
    # --------------------------------------------------------
    config_path = (
        RESULT_DIR /
        "benchmark_config.json"
    )

    if config_path.exists():

        with config_path.open(
            "r",
            encoding="utf-8"
        ) as f:
            config = json.load(f)

        BATCH_SIZE = int(
            config["batch_size"]
        )
        NODE_CHUNK_SIZE = int(
            config["node_chunk_size"]
        )

        print(
            f"\nRESUME benchmark config | "
            f"batch={BATCH_SIZE} | "
            f"chunk={NODE_CHUNK_SIZE}"
        )

    else:

        config = calibrate_common_batch(
            split_samples,
            tokenizer,
            label2id,
            device,
            use_amp,
        )

        BATCH_SIZE = int(
            config["batch_size"]
        )
        NODE_CHUNK_SIZE = int(
            config["node_chunk_size"]
        )

        config.update({
            "dataset": "DRWeibo",
            "window_min": WINDOW_MIN,
            "target_peak_low_gib":
                TARGET_PEAK_LOW_GIB,
            "target_peak_high_gib":
                TARGET_PEAK_HIGH_GIB,
            "warmup_batches":
                WARMUP_BATCHES,
            "inference_batches":
                INFERENCE_BATCHES,
            "training_batches":
                TRAIN_BATCHES,
            "repeats":
                REPEATS,
        })

        with config_path.open(
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                config,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\nSelected common benchmark profile | "
            f"batch={BATCH_SIZE} | "
            f"chunk={NODE_CHUNK_SIZE}"
        )

    aggregate_results()

    # --------------------------------------------------------
    # Per-model / per-repeat resume.
    # --------------------------------------------------------
    for variant in MODELS:

        for repeat_index in range(
            1,
            REPEATS + 1
        ):

            if (
                load_repeat_result(
                    variant,
                    repeat_index
                )
                is not None
            ):
                print(
                    f"SKIP completed | "
                    f"{variant} | "
                    f"repeat={repeat_index}"
                )
                continue

            try:

                benchmark_one_repeat(
                    variant=variant,
                    repeat_index=repeat_index,
                    split_samples=split_samples,
                    tokenizer=tokenizer,
                    label2id=label2id,
                    device=device,
                    use_amp=use_amp,
                    batch_size=BATCH_SIZE,
                    node_chunk_size=NODE_CHUNK_SIZE,
                )

            except Exception as exc:

                if is_cuda_oom(exc):
                    print("\nCUDA OOM during final benchmark.")

                    fallback_batches = [48, 32, 24, 16, 12, 8, 4]
                    smaller = [b for b in fallback_batches if b < BATCH_SIZE]

                    if not smaller:
                        raise

                    new_batch = smaller[0]
                    print(
                        f"Auto fallback: batch {BATCH_SIZE} -> {new_batch}; "
                        f"chunk remains {NODE_CHUNK_SIZE}. "
                        "Saving config and restarting this repeat on next program run."
                    )

                    config["batch_size"] = new_batch

                    with config_path.open(
                        "w",
                        encoding="utf-8"
                    ) as f:
                        json.dump(
                            config,
                            f,
                            indent=2,
                            ensure_ascii=False
                        )

                    cleanup_cuda()

                    # Exit cleanly so rerunning resumes completed repeats and
                    # restarts only the failed repeat with the smaller batch.
                    return

                raise

            aggregate_results()

    summary = aggregate_results()

    print("\n" + "=" * 96)
    print("COMPLEXITY ANALYSIS FINISHED")
    print("=" * 96)

    if summary is not None:
        print(summary.to_string(index=False))

    print(
        f"\nResults directory:\n"
        f"{RESULT_DIR}"
    )

    print("\nKey files:")
    print(
        RESULT_DIR /
        "benchmark_config.json"
    )
    print(
        RESULT_DIR /
        "batch_calibration.csv"
    )
    print(
        RESULT_DIR /
        "all_complexity_runs.csv"
    )
    print(
        RESULT_DIR /
        "complexity_summary.csv"
    )


if __name__ == "__main__":
    main()